# 📊 01 — Exploratory Data Analysis
**Customer Churn Prediction · ChurnGuard**

This notebook performs a comprehensive EDA on the IBM Telco Customer Churn dataset.

---
**Contents**
1. Dataset overview
2. Missing value analysis
3. Target distribution
4. Univariate distributions
5. Churn vs key features
6. Correlation heatmap
7. Key insights summary

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import download_dataset, load_raw_data

# Plotting theme
plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor':   '#1A1D27',
    'axes.edgecolor':   '#2D3142',
    'axes.labelcolor':  '#E2E8F0',
    'xtick.color':      '#94A3B8',
    'ytick.color':      '#94A3B8',
    'text.color':       '#E2E8F0',
    'grid.color':       '#2D3142',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
})
TEAL, CORAL, PURPLE, AMBER = '#00C2CB', '#FF6B6B', '#8B5CF6', '#F59E0B'
PALETTE = {0: TEAL, 1: CORAL}

## 1. Load Dataset

In [ ]:
download_dataset()
df = load_raw_data()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Basic Stats ===')
df.describe(include='all').T

## 2. Missing Value Analysis

In [ ]:
# TotalCharges has spaces that parse as NaN after to_numeric
df['TotalCharges_num'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0]

fig, ax = plt.subplots(figsize=(8, 3))
if not missing_df.empty:
    missing_df['Missing'].plot(kind='barh', ax=ax, color=CORAL)
    ax.set_title('Missing Values per Column', fontsize=13, fontweight='bold')
    for i, (v, pct) in enumerate(zip(missing_df['Missing'], missing_df['Pct'])):
        ax.text(v + 0.2, i, f'{v} ({pct}%)', va='center', fontsize=10, color='#E2E8F0')
else:
    ax.text(0.5, 0.5, 'No missing values (before type casting)',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
plt.tight_layout()
plt.show()
print('\nRows with missing TotalCharges:', df['TotalCharges_num'].isna().sum())

**Insight:** 11 rows have `TotalCharges` stored as a space (' ') — these correspond to brand-new customers with 0 tenure. We impute with median.

## 3. Target Distribution (Churn)

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
bars = ax1.bar(churn_counts.index, churn_counts.values,
               color=[CORAL, TEAL], width=0.5, edgecolor='none')
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax1.set_title('Churn Distribution', fontsize=13, fontweight='bold')
ax1.set_ylabel('Customers')
ax1.grid(axis='y', alpha=0.3)

# Donut
wedges, texts, autotexts = ax2.pie(
    churn_counts.values,
    labels=churn_counts.index,
    autopct='%1.1f%%',
    colors=[CORAL, TEAL],
    startangle=90,
    wedgeprops=dict(width=0.6, edgecolor='#0F1117'),
    textprops=dict(color='#E2E8F0', fontsize=11),
)
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold')
ax2.set_title('Churn Rate', fontsize=13, fontweight='bold')

plt.suptitle('Customer Churn — IBM Telco Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visuals/01_churn_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0F1117')
plt.show()
print(f'Class ratio (Retain:Churn) = {churn_pct["No"]:.1f}% : {churn_pct["Yes"]:.1f}%')

**Insight:** The dataset is imbalanced — 73.5% retained vs 26.5% churned. We'll use SMOTE during training to handle this.

## 4. Univariate Distributions

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges'],
                          [TEAL, PURPLE, AMBER]):
    ax.hist(df[col].dropna(), bins=35, color=color, alpha=0.8, edgecolor='none')
    mean_v = df[col].mean()
    ax.axvline(mean_v, color='white', linestyle='--', linewidth=1.5,
               label=f'Mean: {mean_v:.1f}')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Numeric Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/02_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 5. Churn vs Key Features

In [ ]:
# Churn by contract type
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
cat_cols = ['Contract', 'InternetService', 'PaymentMethod',
            'TechSupport', 'OnlineSecurity', 'PaperlessBilling']

for ax, col in zip(axes.flatten(), cat_cols):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=[TEAL, CORAL], edgecolor='none',
            width=0.6, rot=20)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.legend(title='Churn', labels=['No', 'Yes'], fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Churn Rate by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/03_churn_by_category.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# Boxplots — numeric vs churn
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col, color in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges'],
                          [TEAL, PURPLE, AMBER]):
    groups = [df[df['Churn']=='No'][col].dropna(),
              df[df['Churn']=='Yes'][col].dropna()]
    bp = ax.boxplot(groups, patch_artist=True, widths=0.5,
                    medianprops=dict(color='white', linewidth=2))
    for patch, clr in zip(bp['boxes'], [TEAL, CORAL]):
        patch.set_facecolor(clr); patch.set_alpha(0.75)
    ax.set_xticklabels(['Retained', 'Churned'])
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Numeric Features vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/04_boxplots.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 6. Correlation Heatmap

In [ ]:
# Encode for correlation
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    df_enc[col] = pd.factorize(df_enc[col])[0]
df_enc['TotalCharges'] = pd.to_numeric(df_enc['TotalCharges'], errors='coerce').fillna(0)
df_enc = df_enc.drop(columns=['customerID'], errors='ignore')

corr = df_enc.corr().round(2)

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(10, 180, s=90, l=40, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', linewidths=0.4, linecolor='#0F1117',
            annot_kws={'size': 7}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../visuals/05_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 7. Key Insights

In [ ]:
print('=== Key Findings ===')
print()

# Churn by contract
cc = df.groupby('Contract')['Churn_bin'].mean() * 100
print('Churn rate by Contract:')
for idx, val in cc.items():
    print(f'  {idx}: {val:.1f}%')

print()
# Churn by internet
ci = df.groupby('InternetService')['Churn_bin'].mean() * 100
print('Churn rate by Internet Service:')
for idx, val in ci.items():
    print(f'  {idx}: {val:.1f}%')

print()
# Churn vs senior
cs = df.groupby('SeniorCitizen')['Churn_bin'].mean() * 100
print('Churn rate by Senior Citizen:')
for idx, val in cs.items():
    print(f'  {"Senior" if idx else "Non-senior"}: {val:.1f}%')

print()
print('Average tenure — Churned:', df[df['Churn']=='Yes']['tenure'].mean().round(1))
print('Average tenure — Retained:', df[df['Churn']=='No']['tenure'].mean().round(1))
print()
print('Average MonthlyCharges — Churned:', df[df['Churn']=='Yes']['MonthlyCharges'].mean().round(2))
print('Average MonthlyCharges — Retained:', df[df['Churn']=='No']['MonthlyCharges'].mean().round(2))

## Summary

| Insight | Detail |
|---------|--------|
| **Class imbalance** | 73.5% retained / 26.5% churned → SMOTE needed |
| **Month-to-month contracts** | 42% churn rate vs 11% (1 yr) and 3% (2 yr) |
| **Fiber optic** | 41% churn vs 19% (DSL) and 7% (No internet) |
| **Short tenure** | Churned customers average 18 months vs 38 months retained |
| **High monthly charges** | Churned: $74/mo vs retained $61/mo |
| **Senior citizens** | 41% churn vs 23% for non-seniors |
| **Electronic check** | Highest payment method churn rate at 45% |

➡️ Proceed to **02_feature_engineering.ipynb**